In [1]:
# Gensim 라이브러리
# 대규모 한국어 코퍼스(Wikipedia, Naver News등) 미리학습  Word2Vec, FastText 모델파일 이 들어 있는 라이브러리

In [2]:
%pip install --upgrade gensim

Note: you may need to restart the kernel to use updated packages.


In [3]:
# step1 : 사전학습된 모델로드
import gensim
# FastText 모델 로드
# OOV(Out-Of-Vocabulary, 사전에 없는 단어  ) 문제에 강점이 있다 오타가많은 리뷰데이터 특화

# step2 : 임베딩 행렬 구축
# 로드한 모델 수십만개의 단어 벡터가 있음, 우리가 구성한 단어장(vocab)에 있는 단어만 필요(단어장 크기, 임베딩 차원)크기의 빈행렬생성
# 우리 단어장에 있는 단어가 사전학습한 모델에 존재하면 그 벡터를 복사

# step3 : nn.Embedding에 가중치 덮어쓰기

In [4]:
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [1, 0, 1, 0, 1, 0] # 1: 긍정, 0: 부정
}
df = pd.DataFrame(data)

# 데이터가 매우 작아서 클래스 비율이 유지되도록 직접 분리합니다.
train_df = df.iloc[[0, 1, 2, 3]].reset_index(drop=True)
test_df = df.iloc[[4, 5]].reset_index(drop=True)

def tokenizer(text):
    return text.split()

vocab = {'<PAD>': 0, '<UNK>': 1}
for review in train_df['reviews']:
    for word in tokenizer(review):
        if word not in vocab:
            vocab[word] = len(vocab)

max_sequence_len = 8

def text_to_sequence(text, vocab, max_sequence_len):
    seq = [vocab.get(word, vocab['<UNK>']) for word in tokenizer(text)]
    if len(seq) < max_sequence_len:
        seq += [vocab['<PAD>']] * (max_sequence_len - len(seq))
    return seq[:max_sequence_len]

train_df['sequences'] = train_df['reviews'].apply(
    lambda x: text_to_sequence(x, vocab, max_sequence_len)
)
test_df['sequences'] = test_df['reviews'].apply(
    lambda x: text_to_sequence(x, vocab, max_sequence_len)
)

class ReviewDataSet(Dataset):
    def __init__(self, sequences, labels):
        self.x = torch.LongTensor(sequences)
        self.y = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

train_dataset = ReviewDataSet(
    train_df['sequences'].tolist(),
    train_df['ratings'].tolist()
)
test_dataset = ReviewDataSet(
    test_df['sequences'].tolist(),
    test_df['ratings'].tolist()
)

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    generator=generator
)
test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

# 아래 기존 학습 코드와의 호환을 위해 별칭을 둡니다.
dataloader = train_loader

c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ko.300.vec.gz

In [5]:
from pathlib import Path
from urllib.request import urlretrieve
import gzip
import shutil

url = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ko.300.vec.gz'
gz_path = Path('cc.ko.300.vec.gz')
vec_path = Path('cc.ko.300.vec')

if vec_path.exists():
    print(f'{vec_path} 파일이 이미 있어 다운로드를 생략합니다.')
else:
    if not gz_path.exists():
        print('파일 다운로드....')
        urlretrieve(url, gz_path)
    else:
        print(f'{gz_path} 파일이 이미 있어 다운로드를 생략합니다.')

    print('압축해제......')
    with gzip.open(gz_path, 'rb') as f_in:
        with vec_path.open('wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

    gz_path.unlink(missing_ok=True)
    print('압축해제 및 파일정리 완료')

vec_filename = str(vec_path)

파일 다운로드....
압축해제......
압축해제 및 파일정리 완료


In [6]:
# 모델 로드(로컬 FastText) & 임베딩 행렬 구축
# 전체 cc.ko.300.vec를 gensim으로 모두 로드하면 매우 오래 걸리므로,
# 현재 vocab에 필요한 단어 벡터만 스트리밍으로 읽습니다.
VOCAB_SIZE = len(vocab)
EMBED_DIM = 300  # FastText가 300차원
MODEL_PATH = vec_filename

embedding_matrix = np.random.normal(
    scale=0.01,
    size=(VOCAB_SIZE, EMBED_DIM)
).astype(np.float32)
embedding_matrix[vocab['<PAD>']] = np.zeros((EMBED_DIM,), dtype=np.float32)

needed_words = {word for word in vocab if word not in {'<PAD>', '<UNK>'}}
found_words = set()

with open(MODEL_PATH, encoding='utf-8', errors='ignore') as f:
    header = f.readline().strip().split()
    for line in f:
        parts = line.rstrip().split(' ')
        if len(parts) != EMBED_DIM + 1:
            continue

        word = parts[0]
        if word not in needed_words:
            continue

        embedding_matrix[vocab[word]] = np.asarray(parts[1:], dtype=np.float32)
        found_words.add(word)

        if found_words == needed_words:
            break

oov_words = sorted(needed_words - found_words)
pretrained_weight = torch.tensor(embedding_matrix, dtype=torch.float32)

print(
    f'-> FastText 벡터 준비 완료! '
    f'(vocab: {VOCAB_SIZE}개 / 매칭: {len(found_words)}개 / OOV: {len(oov_words)}개)'
)
if oov_words:
    print('OOV 단어:', oov_words)

-> FastText 벡터 준비 완료! (vocab: 17개 / 매칭: 15개 / OOV: 0개)


In [7]:
# ==========================================
# 4. TextCNN 모델 정의
# ==========================================
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, filter_sizes, num_filters, pretrained_weight=None, freeze_emb=False):
        super(TextCNN, self).__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        # FastText 가중치 덮어씌우기
        if pretrained_weight is not None:
            self.embedding.weight.data.copy_(pretrained_weight)
            self.embedding.weight.requires_grad = not freeze_emb

        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (fs, embed_dim)) for fs in filter_sizes
        ])

        self.fc = nn.Linear(len(filter_sizes) * num_filters, 1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x).unsqueeze(1)
        pooled_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x)).squeeze(3)
            p = F.max_pool1d(c, c.size(2)).squeeze(2)
            pooled_outputs.append(p)

        x_cat = self.dropout(torch.cat(pooled_outputs, dim=1))
        return self.fc(x_cat).squeeze(1)


def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            total_loss += loss.item()
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)

    return total_loss / len(dataloader), correct / total

# ==========================================
# 5. 모델 학습 (Training)
# ==========================================
FILTER_SIZES = [2, 3, 4]
NUM_FILTERS = 100

model = TextCNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    filter_sizes=FILTER_SIZES,
    num_filters=NUM_FILTERS,
    pretrained_weight=pretrained_weight,
    freeze_emb=False # 학습하며 임베딩도 우리 데이터에 맞게 미세조정
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

print('\n--- 본격적인 학습을 시작합니다 ---')
epochs = 10
for epoch in range(epochs):
    total_loss = 0
    model.train()

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print(
        f'Epoch {epoch+1:02d}/{epochs} | '
        f'Train Loss: {train_loss:.4f} | '
        f'Test Loss: {test_loss:.4f} | '
        f'Test Acc: {test_acc:.2f}'
    )

print('--- 학습 완료! ---')


--- 본격적인 학습을 시작합니다 ---
Epoch 01/10 | Train Loss: 0.6938 | Test Loss: 0.6944 | Test Acc: 0.50
Epoch 02/10 | Train Loss: 0.6223 | Test Loss: 0.6929 | Test Acc: 0.50
Epoch 03/10 | Train Loss: 0.5541 | Test Loss: 0.6884 | Test Acc: 0.50
Epoch 04/10 | Train Loss: 0.5153 | Test Loss: 0.6818 | Test Acc: 1.00
Epoch 05/10 | Train Loss: 0.4639 | Test Loss: 0.6740 | Test Acc: 1.00
Epoch 06/10 | Train Loss: 0.3916 | Test Loss: 0.6651 | Test Acc: 1.00
Epoch 07/10 | Train Loss: 0.3383 | Test Loss: 0.6558 | Test Acc: 1.00
Epoch 08/10 | Train Loss: 0.2986 | Test Loss: 0.6455 | Test Acc: 1.00
Epoch 09/10 | Train Loss: 0.2922 | Test Loss: 0.6355 | Test Acc: 1.00
Epoch 10/10 | Train Loss: 0.2264 | Test Loss: 0.6249 | Test Acc: 1.00
--- 학습 완료! ---


## Word2Vec

In [8]:
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

# ==========================================
# 1. 데이터 준비
# ==========================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [5, 1, 4, 2, 5, 1]
}

df = pd.DataFrame(data)

# 평점 이진화
# 4~5점 -> 긍정(1)
# 1~3점 -> 부정(0)
df['ratings'] = df['ratings'].apply(
    lambda x: 1 if x >= 4 else 0
)

train_df = df.iloc[[0, 1, 2, 3]].reset_index(drop=True)
test_df = df.iloc[[4, 5]].reset_index(drop=True)

# ==========================================
# 2. 토큰화
# ==========================================
def tokenize(text):
    return text.split()

tokenized_train_sentences = [
    tokenize(review)
    for review in train_df['reviews']
]

# ==========================================
# 3. Word2Vec 학습
# ==========================================
# 예제 데이터에서 직접 학습한 Word2Vec이므로 엄밀한 의미의 외부 사전학습은 아닙니다.
EMBED_DIM = 100
w2v_model = Word2Vec(
    sentences=tokenized_train_sentences,
    vector_size=EMBED_DIM,
    window=3,
    min_count=1,
    workers=1,
    seed=SEED
)
print('Word2Vec 학습 완료')

# ==========================================
# 4. Vocabulary 생성
# ==========================================
vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}
for sentence in tokenized_train_sentences:
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)

# ==========================================
# 5. 문장 -> 숫자 변환
# ==========================================
MAX_LEN = 10

def text_to_sequence(text, vocab, max_len):
    seq = [
        vocab.get(word, vocab['<UNK>'])
        for word in tokenize(text)
    ]
    if len(seq) < max_len:
        seq += [vocab['<PAD>']] * (max_len - len(seq))
    return seq[:max_len]

train_df['sequence'] = train_df['reviews'].apply(
    lambda x: text_to_sequence(x, vocab, MAX_LEN)
)
test_df['sequence'] = test_df['reviews'].apply(
    lambda x: text_to_sequence(x, vocab, MAX_LEN)
)

# ==========================================
# 6. Embedding Matrix 생성
# ==========================================
embedding_matrix = np.random.normal(
    scale=0.01,
    size=(VOCAB_SIZE, EMBED_DIM)
)
embedding_matrix[0] = np.zeros((EMBED_DIM,))

for word, idx in vocab.items():
    if word in ['<PAD>', '<UNK>']:
        continue

    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

pretrained_weight = torch.FloatTensor(
    embedding_matrix
)

# ==========================================
# 7. Dataset
# ==========================================
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        self.x = torch.LongTensor(sequences)
        self.y = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_dataset = ReviewDataset(
    train_df['sequence'].tolist(),
    train_df['ratings'].tolist()
)
test_dataset = ReviewDataset(
    test_df['sequence'].tolist(),
    test_df['ratings'].tolist()
)

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    generator=generator
)
test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

# ==========================================
# 8. TextCNN 모델
# ==========================================

class TextCNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        filter_sizes,
        num_filters,
        pretrained_weight=None,
        freeze_emb=False
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        if pretrained_weight is not None:
            self.embedding.weight.data.copy_(
                pretrained_weight
            )
            self.embedding.weight.requires_grad = (
                not freeze_emb
            )

        self.convs = nn.ModuleList([
            nn.Conv2d(
                in_channels=1,
                out_channels=num_filters,
                kernel_size=(fs, embed_dim)
            )
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(
            len(filter_sizes) * num_filters,
            1
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.unsqueeze(1)
        pooled_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x))
            c = c.squeeze(3)
            p = F.max_pool1d(
                c,
                kernel_size=c.size(2)
            )
            p = p.squeeze(2)
            pooled_outputs.append(p)

        x = torch.cat(
            pooled_outputs,
            dim=1
        )

        x = self.dropout(x)
        logits = self.fc(x)
        return logits.squeeze(1)


def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            total_loss += loss.item()
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)

    return total_loss / len(dataloader), correct / total

# ==========================================
# 9. 모델 생성
# ==========================================

FILTER_SIZES = [2, 3, 4]
NUM_FILTERS = 20

model = TextCNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    filter_sizes=FILTER_SIZES,
    num_filters=NUM_FILTERS,
    pretrained_weight=pretrained_weight,
    freeze_emb=False
)

print(model)

# ==========================================
# 10. Loss / Optimizer
# ==========================================

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ==========================================
# 11. 학습 및 평가
# ==========================================

EPOCHS = 10

print('\n학습 시작\n')
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(
            logits,
            batch_y
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print(
        f'Epoch: {epoch+1:02d} '
        f'Train Loss: {train_loss:.4f} '
        f'Test Loss: {test_loss:.4f} '
        f'Test Acc: {test_acc:.2f}'
    )

print('\n학습 완료')

# ==========================================
# 12. 추론
# ==========================================

def predict(text):
    model.eval()
    sequence = text_to_sequence(
        text,
        vocab,
        MAX_LEN
    )

    x = torch.LongTensor(sequence).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        prob = torch.sigmoid(logits)
        pred = (prob >= 0.5).float()

    print(f'\n리뷰: {text}')
    print(f'긍정확률: {prob.item():.4f}')

    if pred.item() == 1:
        print('예측: 긍정')
    else:
        print('예측: 부정')

Word2Vec 학습 완료
TextCNN(
  (embedding): Embedding(17, 100, padding_idx=0)
  (convs): ModuleList(
    (0): Conv2d(1, 20, kernel_size=(2, 100), stride=(1, 1))
    (1): Conv2d(1, 20, kernel_size=(3, 100), stride=(1, 1))
    (2): Conv2d(1, 20, kernel_size=(4, 100), stride=(1, 1))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=60, out_features=1, bias=True)
)

학습 시작

Epoch: 01 Train Loss: 0.6984 Test Loss: 0.6929 Test Acc: 0.50
Epoch: 02 Train Loss: 0.6937 Test Loss: 0.6929 Test Acc: 0.50
Epoch: 03 Train Loss: 0.6949 Test Loss: 0.6929 Test Acc: 0.50
Epoch: 04 Train Loss: 0.6934 Test Loss: 0.6928 Test Acc: 0.50
Epoch: 05 Train Loss: 0.6932 Test Loss: 0.6927 Test Acc: 0.50
Epoch: 06 Train Loss: 0.6915 Test Loss: 0.6926 Test Acc: 0.50
Epoch: 07 Train Loss: 0.6838 Test Loss: 0.6924 Test Acc: 0.50
Epoch: 08 Train Loss: 0.6943 Test Loss: 0.6923 Test Acc: 0.50
Epoch: 09 Train Loss: 0.6893 Test Loss: 0.6921 Test Acc: 0.50
Epoch: 10 Train Loss: 0.6756 Test Loss: 0.6919 Test

In [9]:
# ==========================================
# 13. 테스트
# ==========================================

predict('배우 연기가 정말 훌륭하다')
predict('돈 아까운 최악의 영화')


리뷰: 배우 연기가 정말 훌륭하다
긍정확률: 0.5033
예측: 긍정

리뷰: 돈 아까운 최악의 영화
긍정확률: 0.4988
예측: 부정
